Pretrained Tokenizer

In [1]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

Create an input sentence

In [2]:
sentence = "I love Machine Learning"

Tokenize the sentence

In [3]:
tokens = tokenizer.tokenize(sentence)
print(tokens)

['i', 'love', 'machine', 'learning']


Convert sentence into token IDs

In [4]:
token_ids = tokenizer.encode(sentence)
print(token_ids)

[101, 1045, 2293, 3698, 4083, 102]


Convert tokenIDs back into tokens to see the special tokens

In [5]:
decoded_tokens = tokenizer.convert_ids_to_tokens(token_ids)
print(decoded_tokens)

['[CLS]', 'i', 'love', 'machine', 'learning', '[SEP]']


[CLS] => Start of the input

[SEP] => End of the input

Positional Encoding:

sent 1: The dog chased the boy.

sent 2: The boy chased the dog.

Both the sentences contains the similar words but the overall meaning is different because the word order changed. And transformers use positional encoding to understand the word order.

Text -> Tokens ->  Token Embeddings -> Add Positional Ecodings -> Transformer Input

In [6]:
import numpy as np

sentence = "I love NLP"
tokens = tokenizer.tokenize(sentence)

token_embeddings = np.array([
    [0.2, 0.4, 0.1, 0.7],   # Embedding for "I"
    [0.5, 0.1, 0.8, 0.3],   # Embedding for "love"
    [0.9, 0.6, 0.2, 0.4]    # Embedding for "NLP"
])
positional_values = np.array([
    [0.1, 0.1, 0.1, 0.1],   # Position value for token 1
    [0.2, 0.2, 0.2, 0.2],   # Position value for token 2
    [0.3, 0.3, 0.3, 0.3]    # Position value for token 3
])

transformer_input = token_embeddings + positional_values

# Print the final Transformer input
print("Final Transformer Input:")
print(transformer_input)

Final Transformer Input:
[[0.3 0.5 0.2 0.8]
 [0.7 0.3 1.  0.5]
 [1.2 0.9 0.5 0.7]]


Multi Head Attention Mechanisms

In [7]:
queries = np.array([
    [0.2, 0.4, 0.6],   # Query for "I"
    [0.5, 0.1, 0.3],   # Query for "love"
    [0.8, 0.2, 0.7]    # Query for "NLP"
])

print("Query Vectors:")
print(queries)

Query Vectors:
[[0.2 0.4 0.6]
 [0.5 0.1 0.3]
 [0.8 0.2 0.7]]


In [8]:
keys = np.array([
    [0.1, 0.3, 0.5],   # Key for "I"
    [0.6, 0.2, 0.4],   # Key for "love"
    [0.7, 0.8, 0.9]    # Key for "NLP"
])

print("Key Vectors:")
print(keys)

Key Vectors:
[[0.1 0.3 0.5]
 [0.6 0.2 0.4]
 [0.7 0.8 0.9]]


In [9]:
values = np.array([
    [1.0, 0.0, 0.0],   # Value for "I"
    [0.0, 1.0, 0.0],   # Value for "love"
    [0.0, 0.0, 1.0]    # Value for "NLP"
])

print("Value Vectors:")
print(values)

Value Vectors:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


In [10]:
# Calculate attention scores using Query and Key
# Formula idea: Q × K.T
attention_scores = np.dot(queries, keys.T)

print("Attention Scores:")
print(attention_scores)

Attention Scores:
[[0.44 0.44 1.  ]
 [0.23 0.44 0.7 ]
 [0.49 0.8  1.35]]


In [11]:
# Create softmax function
def softmax(scores):
    exp_scores = np.exp(scores)
    return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

# Convert attention scores into attention weights
attention_weights = softmax(attention_scores)

print("Attention Weights:")
print(attention_weights)

Attention Weights:
[[0.26661885 0.26661885 0.46676229]
 [0.2608465  0.32180061 0.41735289]
 [0.2115692  0.28845877 0.49997203]]


Machine Translation

In [12]:
from transformers import MarianMTModel, MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-en-fr"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

In [13]:
text = ["I love learning transformers."]

In [14]:
inputs = tokenizer(
    text,
    return_tensors="pt",
    padding=True
)

print(inputs)

{'input_ids': tensor([[   47,  1779,  3655, 14750,   794,     3,     0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])}


In [15]:
translated = model.generate(
    **inputs,
    max_length=50
)

In [16]:
print(translated)

tensor([[59513,   234,     6, 17711,  7497,    16, 38295,     3,     0]])


In [17]:
result = tokenizer.batch_decode(
    translated,
    skip_special_tokens=True
)

print(result)

["J'adore apprendre les transformateurs."]


Question Answering

In [18]:
from transformers import pipeline

pipe = pipeline(
    "document-question-answering",
    model="impira/layoutlm-document-qa"
)

Loading weights:   0%|          | 0/205 [00:00<?, ?it/s]

In [19]:
!apt-get install tesseract-ocr -y
!pip install pytesseract

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [20]:
!pip install -q pytesseract pillow

In [21]:
import pytesseract
print(pytesseract.get_tesseract_version())

4.1.1


In [22]:
from PIL import Image

image = Image.open("/content/images.png")

# OCR quality is likely poor
result = pipe(
    image=image,
    question="What is in the description?"
)

print(result)

[{'score': 0.14325357973575592, 'answer': 'mimbartrmatorpie', 'start': 18, 'end': 18}]


In [24]:
result = pipe(
    image=image,
    question="What is in the description under projecting?"
)

print(result)

[{'score': 0.08773891627788544, 'answer': '‘niace mimbartrmatorpie', 'start': 17, 'end': 18}]


In [23]:
import transformers
print(transformers.__version__)

5.13.1
